The tabular agent of the [previous section](/aiml-common/lectures/reinforcement-learning/robot-navigation/q-learning-navigation/index) saw the room through 216 coarse states and could choose among five commands. In this section the agent receives all 28 numbers of the [navigation MDP](/aiml-common/lectures/reinforcement-learning/robot-navigation/navigation-mdp/index) and outputs any forward speed and turn rate within the TurtleBot3 Waffle's limits. The method is deep deterministic policy gradient (DDPG) [@Lillicrap2015-md], built on the deterministic policy gradient theorem [@Silver2014-ai]. It is one of the algorithms the [turtlebot3_drlnav](https://github.com/tomasvr/turtlebot3_drlnav) project trains in Gazebo and runs on the real robot.

The simulator, reward and episode rules are unchanged.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

from cycler import cycler
from matplotlib_inline.backend_inline import (
    set_matplotlib_formats,
)

set_matplotlib_formats("svg")
FIG_GRAY = "#808080"
FIG_COLORS = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#cb77a6",
    "#c28600",
    "#1d99e0",
    "#7f7f7f",
]
plt.rcParams.update(
    {
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "savefig.facecolor": "none",
        "savefig.transparent": True,
        "text.color": FIG_GRAY,
        "axes.labelcolor": FIG_GRAY,
        "axes.edgecolor": FIG_GRAY,
        "xtick.color": FIG_GRAY,
        "ytick.color": FIG_GRAY,
        "grid.color": FIG_GRAY,
        "grid.alpha": 0.25,
        "legend.facecolor": "none",
        "legend.edgecolor": FIG_GRAY,
        "legend.framealpha": 0,
        "axes.prop_cycle": cycler(color=FIG_COLORS),
        "svg.hashsalt": "aiml",
        "svg.fonttype": "path",
    }
)
BLUE, ORANGE, GREEN, PINK, GOLD = FIG_COLORS[:5]

In [ ]:
V_MAX, W_MAX = 0.26, 1.82  # m/s, rad/s (Waffle limits)
RADIUS = 0.2  # m, robot footprint
DT = 0.2  # s, control period
N_BEAMS = 24
R_MIN, R_MAX = 0.12, 3.5  # m, lidar span
ARENA = 4.0  # m, side of the square room
GOAL_TOL = 0.25  # m
T_MAX = 200  # steps per episode (40 s)

# Obstacles: (x, y, radius) in meters
OBSTACLES = np.array(
    [
        [1.2, 1.3, 0.25],
        [2.8, 1.1, 0.20],
        [2.0, 2.4, 0.30],
        [3.0, 3.0, 0.20],
    ]
)


def ray_cast(x, y, angles):
    """Lidar range along each world-frame angle."""
    dx, dy = np.cos(angles), np.sin(angles)
    r = np.full(angles.shape, np.inf)
    with np.errstate(divide="ignore", invalid="ignore"):
        for wall, d, p in (
            (0.0, dx, x),
            (ARENA, dx, x),
            (0.0, dy, y),
            (ARENA, dy, y),
        ):
            t = (wall - p) / d
            r = np.minimum(r, np.where(t > 0, t, np.inf))
    for cx, cy, rho in OBSTACLES:
        ox, oy = x - cx, y - cy
        b = ox * dx + oy * dy
        disc = b * b - (ox * ox + oy * oy - rho * rho)
        t = -b - np.sqrt(np.maximum(disc, 0.0))
        r = np.minimum(
            r, np.where((disc >= 0) & (t > 0), t, np.inf)
        )
    return np.clip(r, R_MIN, R_MAX)


def clearance(x, y):
    """Distance from a point to the nearest surface."""
    walls = min(x, ARENA - x, y, ARENA - y)
    centers = np.hypot(OBSTACLES[:, 0] - x, OBSTACLES[:, 1] - y)
    return min(walls, np.min(centers - OBSTACLES[:, 2]))


class NavEnv:
    """TurtleBot3 Waffle driving to a goal in a 4 m room."""

    def __init__(self, seed=0, noise=0.02, collision_cost=20.0):
        self.collision_cost = collision_cost
        # Starts and goals. Each reset also seeds that episode's noise,
        # so episode k is the same whatever the earlier episodes did.
        self.rng = np.random.default_rng(seed)
        self.noise = noise
        self.beam_angles = np.linspace(
            -np.pi, np.pi, N_BEAMS, endpoint=False
        )

    def _free_point(self, margin):
        while True:
            p = self.rng.uniform(margin, ARENA - margin, size=2)
            if clearance(*p) > margin:
                return p

    def reset(self, start=None, goal=None):
        self.noise_rng = np.random.default_rng(self.rng.integers(2**63))
        if goal is None:
            goal = self._free_point(0.4)
        self.goal = np.array(goal, float)
        if start is None:
            while True:
                p = self._free_point(0.4)
                if np.hypot(*(p - self.goal)) > 1.5:
                    break
            start = (*p, self.rng.uniform(-np.pi, np.pi))
        self.pose = np.array(start, float)
        self.prev = np.zeros(2)
        self.t = 0
        self.dist = self.goal_distance()
        return self.observe()

    def goal_distance(self):
        return float(np.hypot(*(self.goal - self.pose[:2])))

    def goal_bearing(self):
        x, y, phi = self.pose
        g = np.arctan2(self.goal[1] - y, self.goal[0] - x) - phi
        return float(np.arctan2(np.sin(g), np.cos(g)))

    def scan(self):
        x, y, phi = self.pose
        return ray_cast(x, y, phi + self.beam_angles)

    def observe(self):
        """The 28 numbers the agent receives."""
        return np.concatenate(
            [
                self.scan() / R_MAX,
                [
                    min(self.dist, R_MAX) / R_MAX,
                    self.goal_bearing() / np.pi,
                    self.prev[0] / V_MAX,
                    self.prev[1] / W_MAX,
                ],
            ]
        ).astype(np.float32)

    def step(self, v, w):
        v = float(np.clip(v, 0.0, V_MAX))
        w = float(np.clip(w, -W_MAX, W_MAX))
        vn = v + self.noise * self.noise_rng.standard_normal()
        wn = w + self.noise * self.noise_rng.standard_normal()
        x, y, phi = self.pose
        x += vn * np.cos(phi) * DT
        y += vn * np.sin(phi) * DT
        phi = np.arctan2(
            np.sin(phi + wn * DT), np.cos(phi + wn * DT)
        )
        self.pose = np.array([x, y, phi])
        self.prev = np.array([v, w])
        self.t += 1
        d = self.goal_distance()
        progress, self.dist = self.dist - d, d
        r = 2.0 * progress - 0.05 * abs(w) / W_MAX - 0.01
        event, terminated = None, False
        if d < GOAL_TOL:
            r, event, terminated = r + 20.0, "goal", True
        elif clearance(x, y) < RADIUS:
            r -= self.collision_cost
            event, terminated = "collision", True
        truncated = not terminated and self.t >= T_MAX
        if truncated:
            event = "timeout"
        info = {"event": event}
        return self.observe(), r, terminated, truncated, info

In [ ]:
def draw_room(ax, env=None):
    ax.add_patch(
        Rectangle(
            (0, 0), ARENA, ARENA, fill=False, ec=FIG_GRAY, lw=1.5
        )
    )
    for cx, cy, rho in OBSTACLES:
        ax.add_patch(
            Circle((cx, cy), rho, color=FIG_GRAY, alpha=0.5, lw=0)
        )
    if env is not None:
        ax.plot(*env.goal, marker="*", ms=16, color=GOLD, ls="")
    ax.set_xlim(-0.1, ARENA + 0.1)
    ax.set_ylim(-0.1, ARENA + 0.1)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")


def draw_robot(ax, pose, color=BLUE):
    x, y, phi = pose
    ax.add_patch(
        Circle((x, y), RADIUS, fill=False, ec=color, lw=1.5)
    )
    ax.plot(
        [x, x + RADIUS * np.cos(phi)],
        [y, y + RADIUS * np.sin(phi)],
        color=color,
        lw=1.5,
    )


def go_to_goal(env):
    b = env.goal_bearing()
    v = V_MAX * max(0.0, np.cos(b))
    return v, float(np.clip(2.0 * b, -W_MAX, W_MAX))


def rollout(env, policy, start=None, goal=None):
    env.reset(start=start, goal=goal)
    poses, rewards = [env.pose.copy()], []
    while True:
        _, r, term, trunc, info = env.step(*policy(env))
        poses.append(env.pose.copy())
        rewards.append(r)
        if term or trunc:
            return np.array(poses), np.array(rewards), info


def success_rate(policy, n=300, seed=99):
    env = NavEnv(seed=seed)
    events = {"goal": 0, "collision": 0, "timeout": 0}
    for _ in range(n):
        *_, info = rollout(env, policy)
        events[info["event"]] += 1
    return {k: v / n for k, v in events.items()}

## A deterministic actor and a critic

With a continuous action, the greedy step of Q-learning, $\max_a Q(s, a)$, becomes an optimization problem over $(v, \omega)$ at every time step. DDPG avoids solving it by training a second network to do it. It keeps two networks:

- The actor $\mu_\theta(s)$ maps the 28-number state to an action. Its output layer is a $\tanh$, so each output lies in $[-1, 1]$, and it is rescaled to $v \in [0, 0.26]$ m/s and $\omega \in [-1.82, 1.82]$ rad/s.
- The critic $Q_w(s, a)$ maps a state and an action to an estimate of the action value $q^\mu(s, a)$ of the actor's policy.

A method that learns a policy and a value function together is an actor-critic method. The [REINFORCE](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce/index) section reached the same structure from the policy side, when it subtracted a learned baseline $v_\pi(s_t)$ from the return.

The actor is deterministic: the same state always gives the same command. That suits a velocity controller, which must send one command every 0.2 s and should send smooth ones. The [REINFORCE](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce/index) and [PPO](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/PPO/index) sections instead use stochastic policies $\pi_\theta(a \mid s)$ and sample from them.

## The deterministic policy gradient

The actor should maximize the expected return from the start states,

$$
J(\theta) = \mathbb E_{s_0 \sim p_0}\big[ v^\mu(s_0) \big],
$$

where $p_0$ is the distribution of start states and $v^\mu$ is the value of following $\mu_\theta$. The deterministic policy gradient theorem [@Silver2014-ai] gives its gradient as an expectation over $\rho^\mu$, the discounted distribution of states visited under $\mu_\theta$:

$$
\nabla_\theta J(\theta) = \mathbb E_{s \sim \rho^\mu}\Big[ \nabla_\theta \mu_\theta(s)\, \nabla_a q^\mu(s, a) \big|_{a = \mu_\theta(s)} \Big].
$$

Read it as the chain rule. The critic says in which direction a small change of $(v, \omega)$ would raise the value, and $\nabla_\theta \mu_\theta(s)$ says how to change the actor's weights to move its output that way. Compare it with the policy gradient theorem of the REINFORCE section, for a stochastic policy:

$$
\nabla_\theta J(\pi_\theta) = \mathbb{E}_{\pi_\theta} \left[ \nabla_\theta \log \pi_\theta (a \mid s)\, Q^{\pi_\theta}(s, a) \right].
$$

That gradient raises the log-probability of each sampled action in proportion to its value. It needs the value only at the actions the robot took, and it averages over many sampled actions to find the direction. The deterministic gradient needs one action per state, but it needs the slope $\nabla_a q^\mu$ of the value with respect to the action. Only a differentiable critic can supply that slope. The log-probability term is gone because a deterministic policy has no distribution over actions to differentiate. DDPG replaces $q^\mu$ with the critic $Q_w$, so the actor's loss is simply $-Q_w(s, \mu_\theta(s))$ averaged over a batch of states.

## Learning the critic

The critic learns by temporal differences, like the Q-table. For a stored transition $(s, a, r, s')$ the target is

$$
y = r + \gamma\,(1 - \mathbb 1[\text{terminal}])\, Q_{w'}\big(s', \mu_{\theta'}(s')\big),
$$

and the critic minimizes $(Q_w(s, a) - y)^2$ over a batch. The difference $y - Q_w(s, a)$ is the TD error of [TD prediction](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/prediction/temporal-difference), and the critic's loss is its square. Compared with the Q-learning target, $\max_a Q(s', a)$ is replaced by the value of the action the actor would take in $s'$. As before, a goal or a collision cuts the bootstrap term and a timeout does not.

In the MDP chapter's terms, the critic does policy evaluation for $\mu$. Its target is a one-sample estimate of the right side of the Bellman expectation equation for action values,

$$
q^\mu(s, a) = \mathcal R^a_s + \gamma \sum_{s'} \mathcal P^a_{ss'} \, q^\mu\big(s', \mu(s')\big),
$$

with the stored next state in place of the sum and a network in place of the table. The actor step is policy improvement. Policy iteration in the [navigation MDP section](/aiml-common/lectures/reinforcement-learning/robot-navigation/navigation-mdp/index) set $\pi'(s) = \arg\max_a q_\pi(s, a)$ exactly. The actor takes one gradient step toward a larger $Q_w(s, \mu_\theta(s))$ instead. DDPG alternates the two steps without running either to convergence, which makes it a form of approximate policy iteration.

$Q_{w'}$ and $\mu_{\theta'}$ are target networks: slowly moving copies of the critic and the actor. Three more pieces keep training stable:

- A replay buffer stores the last 200,000 transitions, and each update draws a random batch of 256 from it. Consecutive steps of one trajectory are strongly correlated, and random batches break that correlation. Reusing old transitions is allowed because DDPG, like Q-learning, is off-policy. The target uses $\mu_{\theta'}(s')$, the action of the current target actor, not the action stored in the buffer. A transition collected by an older, noisier policy is therefore still a valid sample of the dynamics. REINFORCE cannot reuse transitions this way, because its gradient is an expectation over the current policy's own actions.
- After each update the target weights move a fraction $\tau = 0.005$ toward the learned weights, $w' \leftarrow \tau w + (1 - \tau) w'$. The targets $y$ then change slowly instead of chasing the critic they train.
- The deterministic actor never explores by itself, so during training Gaussian noise with standard deviation 0.2 is added to its two outputs before rescaling. Lillicrap et al. used temporally correlated Ornstein-Uhlenbeck noise. Independent noise is simpler and works here. The first 5000 steps use uniformly random actions to fill the buffer.

In [ ]:
import copy
import torch
import torch.nn as nn

torch.set_num_threads(4)
OBS_DIM, ACT_DIM = N_BEAMS + 4, 2


def mlp(n_in, n_out, out=None):
    layers = [
        nn.Linear(n_in, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, n_out),
    ]
    return nn.Sequential(*layers, *([out] if out else []))


def to_command(a):
    """Rescale an actor output in [-1, 1]^2 to (v, omega)."""
    return (a[0] + 1) / 2 * V_MAX, a[1] * W_MAX


class ReplayBuffer:
    def __init__(self, size=200_000):
        self.o = np.zeros((size, OBS_DIM), np.float32)
        self.a = np.zeros((size, ACT_DIM), np.float32)
        self.r = np.zeros(size, np.float32)
        self.o2 = np.zeros((size, OBS_DIM), np.float32)
        self.term = np.zeros(size, np.float32)
        self.size, self.i, self.n = size, 0, 0

    def add(self, o, a, r, o2, term):
        k = self.i
        self.o[k], self.a[k], self.r[k] = o, a, r
        self.o2[k], self.term[k] = o2, term
        self.i = (k + 1) % self.size
        self.n = min(self.n + 1, self.size)

    def sample(self, batch, rng):
        idx = rng.integers(0, self.n, batch)
        return [
            torch.as_tensor(x[idx])
            for x in (self.o, self.a, self.r, self.o2, self.term)
        ]


def soft_update(net, target, tau):
    with torch.no_grad():
        for p, pt in zip(net.parameters(), target.parameters()):
            pt.mul_(1 - tau).add_(tau * p)


def actor_policy(actor):
    """Wrap a trained actor as a policy for rollout()."""

    def policy(env):
        with torch.no_grad():
            a = actor(torch.as_tensor(env.observe())).numpy()
        return to_command(a)

    return policy


def evaluate(
    actor, critic, n=50, seed=123, gamma=0.99, horizon=1000,
    env_kwargs=None,
):
    """Greedy success rate, the critic's estimate of the
    start-state value, and the return actually obtained.

    The critic bootstraps through the timeout, so the return
    keeps summing rewards past it, until a goal, a collision or
    `horizon` steps (gamma**1000 < 1e-4). Success counts only
    goals reached before the timeout."""
    env = NavEnv(seed=seed, **(env_kwargs or {}))
    goals, q_est, returns = 0, [], []
    for _ in range(n):
        o = env.reset()
        with torch.no_grad():
            s = torch.as_tensor(o)[None]
            q_est.append(
                critic(torch.cat([s, actor(s)], 1)).item()
            )
        G, k, event = 0.0, 0, None
        while k < horizon:
            with torch.no_grad():
                a = actor(torch.as_tensor(o)).numpy()
            o, r, term, trunc, info = env.step(*to_command(a))
            G, k = G + gamma**k * r, k + 1
            if event is None and (term or trunc):
                event = info["event"]
            if term:
                break
        goals += event == "goal"
        returns.append(G)
    return goals / n, np.mean(q_est), np.mean(returns)

The training loop follows the description above. Every 10,000 steps it runs the actor without noise on 50 fixed episodes and records three numbers: the fraction that reach the goal, the critic's estimate $Q_w(s_0, \mu_\theta(s_0))$ of the start states' value, and the discounted return the robot actually collected from those starts. If the critic is accurate, the last two should agree. The critic's target does not stop at the 40 s timeout, so its estimate includes rewards the robot would collect after it. To measure the same quantity, the evaluation keeps each episode running past the timeout until a goal or a collision, or for at most 1,000 steps, where $\gamma^{1000}$ is below $10^{-4}$. The success rate still counts only goals reached within 40 s.

In [ ]:
def train_ddpg(
    steps=100_000,
    gamma=0.99,
    tau=0.005,
    batch=256,
    sigma=0.2,
    warmup=5000,
    seed=0,
):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    env = NavEnv(seed=seed)
    actor = mlp(OBS_DIM, ACT_DIM, nn.Tanh())
    critic = mlp(OBS_DIM + ACT_DIM, 1)
    actor_t, critic_t = (
        copy.deepcopy(actor),
        copy.deepcopy(critic),
    )
    opt_a = torch.optim.Adam(actor.parameters(), lr=1e-4)
    opt_c = torch.optim.Adam(critic.parameters(), lr=1e-3)
    buf, log, best = ReplayBuffer(), [], (-1.0, None)
    o = env.reset()
    for t in range(steps):
        if t < warmup:
            a = rng.uniform(-1, 1, ACT_DIM)
        else:
            with torch.no_grad():
                a = actor(torch.as_tensor(o)).numpy()
            a = np.clip(
                a + sigma * rng.standard_normal(ACT_DIM), -1, 1
            )
        o2, r, term, trunc, _ = env.step(*to_command(a))
        buf.add(o, a, r, o2, float(term))
        o = env.reset() if term or trunc else o2
        if t < warmup:
            continue
        O, A, R, O2, TERM = buf.sample(batch, rng)
        with torch.no_grad():
            q_next = critic_t(torch.cat([O2, actor_t(O2)], 1))
            y = R + gamma * (1 - TERM) * q_next.squeeze(1)
        q = critic(torch.cat([O, A], 1)).squeeze(1)
        loss_c = ((q - y) ** 2).mean()
        opt_c.zero_grad()
        loss_c.backward()
        opt_c.step()
        loss_a = -critic(torch.cat([O, actor(O)], 1)).mean()
        opt_a.zero_grad()
        loss_a.backward()
        opt_a.step()
        soft_update(actor, actor_t, tau)
        soft_update(critic, critic_t, tau)
        if (t + 1) % 10_000 == 0:
            rate, q_est, G = evaluate(actor, critic)
            log.append((t + 1, rate, q_est, G))
            if rate > best[0]:
                best = (rate, copy.deepcopy(actor))
    return best[1], np.array(log)


ddpg_actor, ddpg_log = train_ddpg()

In [ ]:
steps_k = ddpg_log[:, 0] / 1000
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax.plot(steps_k, ddpg_log[:, 1], color=GREEN, marker="o")
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("training steps (thousands)")
ax.set_ylabel("greedy success rate")
ax.set_title("DDPG: success without exploration noise")
ax2.plot(
    steps_k,
    ddpg_log[:, 2],
    color=ORANGE,
    marker="o",
    label=r"critic estimate $Q_w(s_0, \mu_\theta(s_0))$",
)
ax2.plot(
    steps_k,
    ddpg_log[:, 3],
    color=BLUE,
    marker="o",
    label="return actually obtained",
)
ax2.set_xlabel("training steps (thousands)")
ax2.set_ylabel("discounted return")
ax2.legend(fontsize=8)
ax2.set_title("What the critic believes and what happens")
plt.tight_layout()
plt.show()

*Left: the fraction of 50 fixed episodes in which the actor, run without noise, reaches the goal. Right: the critic's estimate of the start states' value next to the discounted return the robot actually collected from them.*

The actor learns the task for a while: after 30,000 steps it reaches the goal in 44% of the evaluation episodes. Ten thousand steps later it reaches the goal in none of them, and it stays below a quarter until 90,000 steps. Only the last evaluation, at 100,000 steps, is better, at 60%. The right panel shows why. From 30,000 steps on, the critic's estimate of the start-state value is far above the return the robot collects. At 40,000 and 50,000 steps the critic predicts about 24 for the start states, while the robot collects between $-5$ and $-3$ from them. The gap narrows only at the end of training, where the critic still predicts about 16 against an actual return of about 7. For comparison, the agent of the next section reaches the goal in almost every episode and collects about 15.

This is the overestimation that Fujimoto et al. identified in DDPG [@Fujimoto2018-uc]. The actor is trained to climb the critic, so it plays the role of the max in the value-iteration backup of the [navigation MDP section](/aiml-common/lectures/reinforcement-learning/robot-navigation/navigation-mdp/index). With exact values, the max picks the best action. With noisy estimates, it tends to pick the action whose error happens to be largest, so the maximum of the estimates is biased upward even when each estimate is unbiased. Wherever the critic overestimates an action, the actor moves toward that action, and the bootstrapped targets pass the error back to earlier states. Nothing in the loop pulls the estimate back down, because the agent rarely tries the alternative actions that would show the critic is wrong. The run above produced one collapse and a partial recovery at the end. Other seeds and step sizes give different curves, often with several rises and collapses, and the best and worst moments depend on luck.

The training loop kept the actor from its best evaluation. On the 300 test episodes of the earlier sections:

In [ ]:
print(
    "DDPG (best checkpoint):",
    success_rate(actor_policy(ddpg_actor)),
)

The best DDPG checkpoint, picked by an evaluation on 50 episodes, does worse than the 216-state Q-table of the previous section, even though it sees the full lidar scan and steers with continuous commands. Keeping the best checkpoint hides the instability without removing it. You cannot ship a training method whose result depends on when you stop it. The [next section](/aiml-common/lectures/reinforcement-learning/robot-navigation/td3-navigation/index) changes how the critic's targets are built and gets a stable learner from the same networks.